In [21]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.append(str(repo_root))

import pandas as pd
import src.utils.pdata_io as pdio

from src.qc.qc_events import load_behavior_qc_tables
from src.proc.extract_epoch_windows import (
    build_prepost_epoch_windows,
    save_epoch_windows,
    load_epoch_windows,
)

data_root, pdata_root, cc_data = pdio.load_project_context()

events_df, session_summary_df = load_behavior_qc_tables(
    pdata_root=pdata_root,
    filename="behavior_QC.h5"
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Behavior QC tables: /mnt/pdata/Classical_Conditioning/_cache/behavior_QC.h5


In [12]:
session_summary_df


,animal,date,phase,fs,n_samples,recording_duration_s,short_recording,n_onsets,n_offsets,n_pairs,...,rig_day_from_date,rig_session_number,phase_day_from_date,phase_session_number,rig_session_start_s,rig_session_start_min,phase_session_start_s,phase_session_start_min,rig_calendar_day,phase_calendar_day
0,NML_04,2025_12_27,habituation,NaN,NaN,NaN,False,NaN,NaN,NaN,...,1,1,1,1,0.0,0.000000,0.0,0.0,1,1
1,NML_04,2025_12_28,habituation,NaN,NaN,NaN,False,NaN,NaN,NaN,...,2,2,2,2,0.0,0.000000,0.0,0.0,2,2
2,NML_04,2025_12_29,habituation,NaN,NaN,NaN,False,NaN,NaN,NaN,...,3,3,3,3,0.0,0.000000,0.0,0.0,3,3
3,NML_04,2025_12_30,habituation,NaN,NaN,NaN,False,NaN,NaN,NaN,...,4,4,4,4,0.0,0.000000,0.0,0.0,4,4
4,NML_04,2025_12_31,habituation,NaN,NaN,NaN,False,NaN,NaN,NaN,...,5,5,5,5,0.0,0.000000,0.0,0.0,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,NML_08,2026_03_29,unknown,NaN,NaN,NaN,False,NaN,NaN,NaN,...,56,54,7,7,34872.4,581.206667,0.0,0.0,56,7
268,NML_08,2026_03_30,unknown,NaN,NaN,NaN,False,NaN,NaN,NaN,...,57,55,8,8,34872.4,581.206667,0.0,0.0,57,8
269,NML_08,2026_03_31,unknown,NaN,NaN,NaN,False,NaN,NaN,NaN,...,58,56,9,9,34872.4,581.206667,0.0,0.0,58,9
270,NML_08,2026_04_01,unknown,NaN,NaN,NaN,False,NaN,NaN,NaN,...,59,57,10,10,34872.4,581.206667,0.0,0.0,59,10


In [22]:
cols = [
    "animal",
    "date",
    "phase",

    "rig_day_from_date",
    "rig_session_number",

    "phase_day_from_date",
    "phase_session_number",

    "rig_session_start_min",
    "phase_session_start_min",
    "recording_duration_min",
]

session_summary_df[cols].sort_values(
    ["animal", "rig_session_number"]
)
# session_summary_df

,animal,date,phase,rig_day_from_date,rig_session_number,phase_day_from_date,phase_session_number,rig_session_start_min,phase_session_start_min,recording_duration_min
0,NML_04,2025_12_27,habituation,1,1,1,1,0.000000,0.000000,21.846667
1,NML_04,2025_12_28,habituation,2,2,2,2,21.846667,21.846667,24.556667
2,NML_04,2025_12_29,habituation,3,3,3,3,46.403333,46.403333,20.908333
3,NML_04,2025_12_30,habituation,4,4,4,4,67.311667,67.311667,27.070000
4,NML_04,2025_12_31,habituation,5,5,5,5,94.381667,94.381667,22.110000
...,...,...,...,...,...,...,...,...,...,...
267,NML_08,2026_03_29,unknown,56,54,7,7,985.985000,0.000000,NaN
268,NML_08,2026_03_30,unknown,57,55,8,8,985.985000,0.000000,NaN
269,NML_08,2026_03_31,unknown,58,56,9,9,985.985000,0.000000,NaN
270,NML_08,2026_04_01,unknown,59,57,10,10,985.985000,0.000000,NaN


In [23]:
known_phases = ["habituation", "air_training", "tone_air_training"]

check_cols = [
    "animal",
    "date",
    "phase",
    "rig_day_from_date",
    "rig_session_number",
    "phase_day_from_date",
    "phase_session_number",
    "rig_session_start_min",
    "phase_session_start_min",
    "recording_duration_min",
    "status",
]

# 1. Known-phase sessions with missing duration
missing_duration = session_summary_df[
    session_summary_df["phase"].isin(known_phases) &
    session_summary_df["recording_duration_min"].isna()
][check_cols]

missing_duration

,animal,date,phase,rig_day_from_date,rig_session_number,phase_day_from_date,phase_session_number,rig_session_start_min,phase_session_start_min,recording_duration_min,status
177,NML_07,2026_02_23,air_training,22,22,3,3,442.431667,43.996667,NaN,error: 'NoneType' object is not subscriptable
235,NML_08,2026_02_23,air_training,22,22,3,3,470.141667,44.000000,NaN,error: 'NoneType' object is not subscriptable


In [24]:
# 2. Very short known-phase sessions
short_sessions = session_summary_df[
    session_summary_df["phase"].isin(known_phases) &
    session_summary_df["recording_duration_min"].notna() &
    (session_summary_df["recording_duration_min"] < 10)
][check_cols]

short_sessions

,animal,date,phase,rig_day_from_date,rig_session_number,phase_day_from_date,phase_session_number,rig_session_start_min,phase_session_start_min,recording_duration_min,status
185,NML_07,2026_03_04,air_training,31,30,12,11,592.610000,194.175000,0.111667,ok
199,NML_07,2026_03_18,tone_air_training,45,44,7,7,864.933333,125.383333,3.861667,ok


In [25]:
# 3. Check cumulative rig-time consistency
tmp = session_summary_df.sort_values(
    ["animal", "rig_session_number"]
).copy()

tmp["expected_rig_start_min"] = (
    tmp.groupby("animal")["recording_duration_min"]
    .apply(lambda x: x.fillna(0).cumsum().shift(fill_value=0))
    .reset_index(level=0, drop=True)
)

tmp["rig_start_error_min"] = (
    tmp["rig_session_start_min"] - tmp["expected_rig_start_min"]
)

rig_time_errors = tmp[
    tmp["rig_start_error_min"].abs() > 1e-6
][check_cols + ["expected_rig_start_min", "rig_start_error_min"]]

rig_time_errors

,animal,date,phase,rig_day_from_date,rig_session_number,phase_day_from_date,phase_session_number,rig_session_start_min,phase_session_start_min,recording_duration_min,status,expected_rig_start_min,rig_start_error_min


In [26]:
# 4. Check cumulative phase-time consistency
tmp = session_summary_df.sort_values(
    ["animal", "phase", "phase_session_number"]
).copy()

tmp["expected_phase_start_min"] = (
    tmp.groupby(["animal", "phase"])["recording_duration_min"]
    .apply(lambda x: x.fillna(0).cumsum().shift(fill_value=0))
    .reset_index(level=[0, 1], drop=True)
)

tmp["phase_start_error_min"] = (
    tmp["phase_session_start_min"] - tmp["expected_phase_start_min"]
)

phase_time_errors = tmp[
    tmp["phase_start_error_min"].abs() > 1e-6
][check_cols + ["expected_phase_start_min", "phase_start_error_min"]]

phase_time_errors

,animal,date,phase,rig_day_from_date,rig_session_number,phase_day_from_date,phase_session_number,rig_session_start_min,phase_session_start_min,recording_duration_min,status,expected_phase_start_min,phase_start_error_min


In [27]:
known_phases = ["habituation", "air_training", "tone_air_training"]

session_summary_df["good_session_basic"] = (
    session_summary_df["phase"].isin(known_phases)
    & session_summary_df["recording_duration_min"].notna()
    & (session_summary_df["recording_duration_min"] >= 10)
    & (session_summary_df["status"] == "ok")
)

In [28]:
good_cols = [
    "animal",
    "date",
    "phase",
    "good_session_basic",
    "recording_duration_min",
    "rig_session_start_min",
    "phase_session_start_min",
]

events_df = events_df.merge(
    session_summary_df[good_cols],
    on=["animal", "date", "phase"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_session")
)

In [29]:
events_for_epochs = events_df[
    events_df["good_session_basic"] == True
].copy()

In [30]:
events_for_epochs

,animal,date,phase,event_number,on_idx,off_idx,on_time_s,off_time_s,on_duration_s,trial_forward_cm,...,session_time_s,session_time_min,rig_exposure_time_s,rig_exposure_time_min,phase_exposure_time_s,phase_exposure_time_min,good_session_basic,recording_duration_min,rig_session_start_min_session,phase_session_start_min_session
0,NML_04,2026_01_12,air_training,0,0,24352,0.000000,4.870400,4.870400,6.333451,...,0.000000,0.000000,21798.400000,363.306667,0.000000,0.000000,True,21.815,363.306667,0.000000
1,NML_04,2026_01_12,air_training,1,99782,119895,19.956400,23.979000,4.022600,5.328141,...,19.956400,0.332607,21818.356400,363.639273,19.956400,0.332607,True,21.815,363.306667,0.000000
2,NML_04,2026_01_12,air_training,2,195327,206396,39.065399,41.279202,2.213802,5.101946,...,39.065399,0.651090,21837.465399,363.957757,39.065399,0.651090,True,21.815,363.306667,0.000000
3,NML_04,2026_01_12,air_training,3,281824,291001,56.364799,58.200199,1.835400,5.403539,...,56.364799,0.939413,21854.764799,364.246080,56.364799,0.939413,True,21.815,363.306667,0.000000
4,NML_04,2026_01_12,air_training,4,366433,377115,73.286598,75.422997,2.136398,5.177345,...,73.286598,1.221443,21871.686598,364.528110,73.286598,1.221443,True,21.815,363.306667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8660,NML_08,2026_03_21,tone_air_training,44,5566095,5603409,1113.218994,1120.681763,7.462769,54.990438,...,1113.218994,18.553650,59039.618994,983.993650,12301.618994,205.026983,True,20.545,965.440000,186.473333
8661,NML_08,2026_03_21,tone_air_training,45,5693327,5731803,1138.665405,1146.360596,7.695190,54.990438,...,1138.665405,18.977757,59065.065405,984.417757,12327.065405,205.451090,True,20.545,965.440000,186.473333
8662,NML_08,2026_03_21,tone_air_training,46,5821719,5859229,1164.343750,1171.845825,7.502075,54.990438,...,1164.343750,19.405729,59090.743750,984.845729,12352.743750,205.879063,True,20.545,965.440000,186.473333
8663,NML_08,2026_03_21,tone_air_training,47,5949145,5991627,1189.828979,1198.325439,8.496460,54.990438,...,1189.828979,19.830482,59116.228979,985.270483,12378.228979,206.303816,True,20.545,965.440000,186.473333


In [31]:
windows_df = build_prepost_epoch_windows(
    events_df,
    window_s=1.0,
    add_pseudo_tone=True,
    add_mid_phase_anchors=True,
    check_overlap=True,
    invalidate_overlaps=False,
)

In [32]:
save_epoch_windows(
    windows_df,
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

[SAVED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s


PosixPath('/mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5')

In [3]:
windows_prepost_1s = build_prepost_epoch_windows(
    events_df,
    window_s=1.0
)

windows_prepost_1s.head()

,animal,date,phase,event_number,anchor_name,anchor_type,anchor_time_s,window_position,window_s,epoch_name,...,session_duration_s,session_time_bin,parent_event_valid,valid_window_basic,valid_window,overlap_flag,overlap_s,overlap_with,anchor_warning,invalid_reason
0,NML_04,2026_01_12,air_training,0,pseudo_tone_on,pseudo,-3.0,post,1.0,pseudo_tone_on_post_1s,...,1308.9,early,False,False,False,False,0.0,,,parent_event_invalid
1,NML_04,2026_01_12,air_training,0,pseudo_tone_on,pseudo,-3.0,pre,1.0,pseudo_tone_on_pre_1s,...,1308.9,early,False,False,False,False,0.0,,,parent_event_invalid
2,NML_04,2026_01_12,air_training,0,air_on,main,0.0,post,1.0,air_on_post_1s,...,1308.9,early,False,False,False,False,0.0,,,parent_event_invalid
3,NML_04,2026_01_12,air_training,0,air_on,main,0.0,pre,1.0,air_on_pre_1s,...,1308.9,early,False,False,False,False,0.0,,,parent_event_invalid
4,NML_04,2026_01_12,air_training,0,pseudo_tone_off,pseudo,2.0,post,1.0,pseudo_tone_off_post_1s,...,1308.9,early,False,False,False,False,0.0,,,parent_event_invalid


In [4]:
windows_prepost_1s.groupby(
    ["phase", "epoch_name", "valid_window"]
).size().reset_index(name="n")

,phase,epoch_name,valid_window,n
0,air_training,air_off_mid_post_1s,False,157
1,air_training,air_off_mid_post_1s,True,3379
2,air_training,air_off_mid_pre_1s,False,157
3,air_training,air_off_mid_pre_1s,True,3379
4,air_training,air_off_post_1s,False,698
...,...,...,...,...
67,tone_air_training,tone_off_pre_1s,True,1362
68,tone_air_training,tone_on_post_1s,False,247
69,tone_air_training,tone_on_post_1s,True,1685
70,tone_air_training,tone_on_pre_1s,False,247


In [7]:
save_epoch_windows(
    windows_prepost_1s,
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

[SAVED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s


PosixPath('/mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5')

In [6]:
cache = Path(pdata_root) / "_cache"
cache.mkdir(parents=True, exist_ok=True)

encoder_metrics_file = cache / "behavior_epoch_metrics.h5"

encoder_epoch_df.to_hdf(
    encoder_metrics_file,
    key="encoder/window_metrics_1s",
    mode="w",
    format="table"
)

print("Saved:", encoder_metrics_file)

NameError: name 'encoder_epoch_df' is not defined

In [6]:
encoder_epoch_df.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

,phase,epoch_name,n_windows
0,air_training,air_off_center,3380
1,air_training,air_on_center,3380
2,habituation,LED_off_center,2915
3,habituation,LED_on_center,2915
4,tone_air_training,air_off_center,1685
5,tone_air_training,air_on_center,1685
6,tone_air_training,tone_off_center,1685
7,tone_air_training,tone_on_center,1685


In [7]:
summary_speed = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        mean_speed=("mean_speed_path_cms", "mean"),
        sem_speed=("mean_speed_path_cms", lambda x: x.std() / np.sqrt(len(x))),
        n=("mean_speed_path_cms", "count")
    )
    .reset_index()
)

summary_speed

,phase,epoch_name,mean_speed,sem_speed,n
0,air_training,air_off_center,6.226388,0.183213,3380
1,air_training,air_on_center,1.705310,0.034726,3380
2,habituation,LED_off_center,1.719026,0.064177,2915
3,habituation,LED_on_center,1.400333,0.052130,2915
4,tone_air_training,air_off_center,4.651130,0.039276,1685
5,tone_air_training,air_on_center,1.015747,0.024942,1685
6,tone_air_training,tone_off_center,6.265828,0.070467,1685
7,tone_air_training,tone_on_center,0.587699,0.027133,1685


In [8]:
state_summary = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        frac_stationary=("frac_stationary", "mean"),
        frac_forward=("frac_forward", "mean"),
        frac_backward=("frac_backward", "mean"),
        frac_low_net=("frac_low_net_movement", "mean"),
        n=("frac_stationary", "count")
    )
    .reset_index()
)

state_summary

,phase,epoch_name,frac_stationary,frac_forward,frac_backward,frac_low_net,n
0,air_training,air_off_center,0.067437,0.898347,0.034216,0.0,3380
1,air_training,air_on_center,0.595183,0.353475,0.051342,0.0,3380
2,habituation,LED_off_center,0.742896,0.226815,0.030289,0.0,2915
3,habituation,LED_on_center,0.764472,0.199824,0.035704,0.0,2915
4,tone_air_training,air_off_center,0.098344,0.845463,0.056193,0.0,1685
5,tone_air_training,air_on_center,0.690426,0.277400,0.032174,0.0,1685
6,tone_air_training,tone_off_center,0.081787,0.912386,0.005827,0.0,1685
7,tone_air_training,tone_on_center,0.838745,0.142160,0.019095,0.0,1685


In [9]:
tone_df = encoder_epoch_df[
    encoder_epoch_df["phase"] == "tone_air_training"
].copy()

tone_df.groupby("epoch_name").agg(
    mean_path_speed=("mean_speed_path_cms", "mean"),
    frac_forward=("frac_forward", "mean"),
    frac_stationary=("frac_stationary", "mean"),
    n=("mean_speed_path_cms", "count")
).reset_index()

,epoch_name,mean_path_speed,frac_forward,frac_stationary,n
0,air_off_center,4.651130,0.845463,0.098344,1685
1,air_on_center,1.015747,0.277400,0.690426,1685
2,tone_off_center,6.265828,0.912386,0.081787,1685
3,tone_on_center,0.587699,0.142160,0.838745,1685
